# Lasso Regression Code Companion

This notebook connects Lasso Regression code with the theory: Linear Regression plus an L1 penalty that can shrink some coefficients exactly to zero.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Load a Regression Dataset

Lasso is used for regression. It is especially useful when we want regularization and feature selection.


In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print("Rows and features:", X.shape)
X.head()


## Split the Data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Train Linear Regression, Lasso, and Elastic Net

Scaling is important because Lasso penalizes coefficient size. Elastic Net is included briefly because it combines L1 and L2 regularization.


In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Lasso Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=1.0, max_iter=10000))
    ]),
    "Elastic Net": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
    ])
}

for model in models.values():
    model.fit(X_train, y_train)


## Evaluate the Models


In [ ]:
rows = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

pd.DataFrame(rows)


## Inspect Feature Selection

A coefficient equal to zero means Lasso removed that feature from the prediction equation.


In [ ]:
lasso_coef = models["Lasso Regression"].named_steps["model"].coef_

coef_table = pd.DataFrame({
    "feature": X.columns,
    "lasso_coefficient": lasso_coef,
    "selected_by_lasso": lasso_coef != 0
}).sort_values("lasso_coefficient", key=abs, ascending=False)

print("Selected features:", int((lasso_coef != 0).sum()))
coef_table


## Effect of Alpha

Larger `alpha` means stronger L1 penalty. As `alpha` grows, more coefficients can become zero.


In [ ]:
rows = []
for alpha in [0.001, 0.01, 0.1, 1, 10]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=10000))
    ])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    coef = model.named_steps["model"].coef_
    rows.append({
        "alpha": alpha,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
        "selected_features": int((coef != 0).sum())
    })

pd.DataFrame(rows)
